# The Racetrack POMDP: state, observation and reward

`RacetrackPOMDP` wraps HighwayEnv's `racetrack-v0` as a **matched pair**: two arms that
share one dynamics path and differ only in what the agent is allowed to see. That is the
whole reason it exists. Every other driving environment in this package is partially
observed by construction, so a planner's performance drop could never be attributed to
partial observability alone rather than to a change of dynamics, reward or map.

This notebook covers three things: what the **state** is, what the **observation** is in
each arm, and how the **reward** is defined. It needs `highway-env`, which is a
development dependency (`pip install -e ".[dev]"`).

In [1]:
import numpy as np

from POMDPPlanners.environments.racetrack_pomdp import (
    ObservationMode,
    RacetrackPOMDP,
    build_racetrack_config,
    racetrack_reward,
)
from POMDPPlanners.environments.racetrack_pomdp.racetrack_schema import (
    AGENT_SLOT_WIDTH,
    EGO_STATE_WIDTH,
    state_agent_rows,
)

np.set_printoptions(precision=3, suppress=True)

## 1. The state

The state is **identical in both arms** — that is what makes the comparison controlled.
It is a flat float vector:

```
[x, y, heading, speed, lat, ang, curvature] + max_tracked_agents x [present, rel_x, rel_y, rel_vx, rel_vy]
```

| Slot | Meaning |
| --- | --- |
| `x`, `y` | Ego position in the map frame, metres |
| `heading` | Ego heading, radians |
| `speed` | Scalar speed along the heading, m/s |
| `lat` | Signed lateral offset from the lane centreline, metres |
| `ang` | Angle between heading and lane direction, radians |
| `curvature` | Signed curvature of the current lane, 1/m (`0` on a straight) |

Each agent slot is in the **ego body frame** — `rel_x` forward, `rel_y` left — and holds
**relative** velocity. Relative rather than absolute because that is exactly what
differencing two ego-aligned occupancy grids can measure; storing an absolute speed would
force the belief to invent an ego-motion correction it cannot observe.

The world state and the planner model's state have the **same width**, deliberately. The
CARLA world is wider than its model, which makes its agent-slot reshape unsafe against a
world vector; nothing here needs extra slots because crash and off-road flags are
transients of a step, not properties of a state, and travel on the metrics channel.

In [2]:
world = RacetrackPOMDP(discount_factor=0.95, observation_mode=ObservationMode.POMDP, seed=3)
state = world.initial_state_dist().sample()[0]

print('state width :', state.shape[0], f'= {EGO_STATE_WIDTH} ego + {world.max_tracked_agents} x {AGENT_SLOT_WIDTH} agent slots')
print('ego block   :', state[:EGO_STATE_WIDTH])
print()
print('agent slots [present, rel_x, rel_y, rel_vx, rel_vy]:')
print(state_agent_rows(state, world.max_tracked_agents))

2026-08-09 20:45:10,741 - INFO: /home/kobi/Documents/github/POMDPPlanners-racetrack/POMDPPlanners/core/environment/environment.py:188 - Initializing RacetrackPOMDP-pomdp environment with discount factor 0.95


state width : 27 = 7 ego + 4 x 5 agent slots
ego block   : [69.1043152  5.         0.        10.         0.         0.
  0.       ]

agent slots [present, rel_x, rel_y, rel_vx, rel_vy]:
[[  1.          52.17347511 -11.8753804   -5.93321266  -6.59312428]
 [  0.           0.           0.           0.           0.        ]
 [  0.           0.           0.           0.           0.        ]
 [  0.           0.           0.           0.           0.        ]]


/home/kobi/Documents/github/POMDPPlanners/.venv/lib/python3.12/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment racetrack-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


Note the ego starts perfectly lane-centred (`lat = 0`) on a straight (`curvature = 0`).
The one opponent is present but tens of metres away — remember that number, it matters in
section 2.

## 2. The observation

This is the only thing that differs between the arms. Both configurations are assembled by
the same function and are equal on every key except `observation` — asserted by a test,
not left as a convention.

In [3]:
mdp_cfg = build_racetrack_config(ObservationMode.MDP)
pomdp_cfg = build_racetrack_config(ObservationMode.POMDP)

differing = [k for k in set(mdp_cfg) | set(pomdp_cfg) if mdp_cfg.get(k) != pomdp_cfg.get(k)]
print('config keys that differ:', differing)
print()
print('shared dynamics keys:')
for key in ('action', 'simulation_frequency', 'policy_frequency', 'speed_limit'):
    print(f'  {key}: {pomdp_cfg[key]}')

config keys that differ: ['observation']

shared dynamics keys:
  action: {'type': 'ContinuousAction', 'longitudinal': True, 'lateral': True}
  simulation_frequency: 15
  policy_frequency: 5
  speed_limit: 10.0


### The fully-observed arm: `Kinematics`

A `(max_tracked_agents + 1, 5)` table of `[presence, x, y, vx, vy]`, **absolute** and
**unnormalised**, ego first. Both of those are explicit overrides: highway-env defaults to
relative, normalised rows, which is not the baseline this comparison needs.

What is still withheld: the other vehicles' **driver policy**. They are `IDMVehicle`s whose
car-following parameters are never observable. So this arm is a *near*-MDP, and a gap
measured against it is a **lower bound** on the cost of partial observability.

In [4]:
mdp_world = RacetrackPOMDP(discount_factor=0.95, observation_mode=ObservationMode.MDP, seed=3)
mdp_world.initial_state_dist().sample()
mdp_obs = mdp_world.initial_observation_dist().sample()[0]

print('shape:', mdp_obs.shape, mdp_obs.dtype)
print('row 0 is the ego  :', mdp_obs[0])
print('row 1 is the other:', mdp_obs[1])

2026-08-09 20:45:10,928 - INFO: /home/kobi/Documents/github/POMDPPlanners-racetrack/POMDPPlanners/core/environment/environment.py:188 - Initializing RacetrackPOMDP-mdp environment with discount factor 0.95


shape: (5, 5) float32
row 0 is the ego  : [ 1.      69.10432  5.      10.       0.     ]
row 1 is the other: [  1.        121.277794   -6.8753805   4.0667872  -6.5931244]


### The partially-observed arm: `OccupancyGrid`

A `(2, 12, 12)` float32 grid over a **±18 m window at 3 m resolution**, aligned to the
ego's own axes so it rotates with the car. Layer 0 is `presence`, layer 1 is `on_road`.

Withheld relative to the baseline: **every velocity, every vehicle identity, and everything
outside the window**.

Three facts about this grid are easy to get wrong and were verified empirically against
highway-env 1.12.1 rather than read off the source:

1. **The ego is drawn into the grid**, always at centre cell `(6, 6)`. Anything reading the
   grid has to drop it — the belief's tracker does so explicitly, otherwise it would track a
   permanent stationary blob at the centre.
2. **Axis 0 is along-track, axis 1 across-track.** Getting this backwards produces a
   plausible but transposed world.
3. **A vehicle marks exactly one cell**, not its 5x2 m footprint.

In [5]:
pomdp_obs = world.initial_observation_dist().sample()[0]
presence = pomdp_obs[0]

print('shape:', pomdp_obs.shape, pomdp_obs.dtype)
print('lit presence cells (along, across):', [tuple(map(int, c)) for c in np.argwhere(presence > 0.5)])
print()
rows = state_agent_rows(state, world.max_tracked_agents)
occupied = rows[rows[:, 0] > 0.5]
print('nearest true opponent range: %.1f m' % np.min(np.linalg.norm(occupied[:, 1:3], axis=1)))

shape: (2, 12, 12) float32
lit presence cells (along, across): [(6, 6)]

nearest true opponent range: 53.5 m


**One lit cell, and it is the ego's own.** The opponent is real and in the state, but it is
far outside the ±18 m window, so the observation says nothing about it.

That is not a quirk of this seed. Sweeping the opponent count over 8 seeds, an opponent is
inside the window on only **15% of steps** at the shipped one opponent, rising to **23%** at
ten. The lap is ~350 m of centreline and the ego covers about 2 m per decision, so most
episodes never meet anyone.

**Consequence worth knowing before you use this environment:** a planner-driven MDP-vs-POMDP
comparison on the default configuration will probably measure nothing, because the two arms
only differ on about one step in seven. Make the discriminating case common first — the
cheapest lever is spawning opponents near the ego rather than raising their count.

## 3. The reward

Reproduced in closed form from highway-env's own `RacetrackEnv._reward`, and shared by the
world and the planner's model so the planner cannot be optimising a different objective:

```
centering = lane_centering_reward / (1 + lane_centering_cost * lat**2)
raw       = centering + action_reward * ||action|| + collision_reward * crashed
reward    = lmap(raw, [collision_reward, 1], [0, 1]) * on_road
```

with the shipped weights `lane_centering_reward = 1`, `lane_centering_cost = 4`,
`action_reward = -0.3`, `collision_reward = -1`.

Two upstream details are reproduced rather than tidied: the normalisation maps from the
literal `1`, not from `lane_centering_reward`; and it does **not clip**, so unusual weights
can push the result outside `[0, 1]`.

In [6]:
cases = [
    ('lane-centred, coasting',      0.0, (0.0, 0.0), False, True),
    ('0.5 m off centre',            0.5, (0.0, 0.0), False, True),
    ('1.0 m off centre',            1.0, (0.0, 0.0), False, True),
    ('centred, full throttle',      0.0, (1.0, 0.0), False, True),
    ('centred, full steering lock', 0.0, (0.0, 1.0), False, True),
    ('centred, both maxed',         0.0, (1.0, 1.0), False, True),
    ('crashed while centred',       0.0, (0.0, 0.0), True,  True),
    ('off the road',                0.0, (0.0, 0.0), False, False),
]
for label, lat, action, crashed, on_road in cases:
    print(f'{label:<28} {racetrack_reward(lat, action, crashed, on_road):.4f}')

lane-centred, coasting       1.0000
0.5 m off centre             0.7500
1.0 m off centre             0.6000
centred, full throttle       0.8500
centred, full steering lock  0.8500
centred, both maxed          0.7879
crashed while centred        0.5000
off the road                 0.0000


### The asymmetry to notice

**A crash scores 0.5. Going off the road scores 0.0.** Leaving the track is punished *twice
as hard* as hitting another car, because `on_road` multiplies the whole normalised reward to
zero while a collision only shifts it down the `[-1, 1]` range before normalisation.

That is highway-env's own design, not something introduced here, but it shapes behaviour: a
planner facing an unavoidable choice will prefer the collision. Anyone using this
environment for risk-sensitive or severity work should decide whether that ordering is what
they want before reading anything into the results.

Because the reward is normalised into `[0, 1]`, planner exploration constants tuned for
unbounded rewards are badly miscalibrated here. A UCB constant of 5-6 makes PFT-DPW explore
almost uniformly and drive off the road; `~0.3` survives roughly five times longer on a
fifth of the search budget.

### Severity, not just occurrence

A collision rate cannot separate a scrape from a high-speed impact, and the mean speed over
an episode averages the impact away. So the environment also reports `collision_speed_mps`:
the ego's speed on the crashing step, reduced with `MAX` over the episode. This works
because highway-env applies its crash braking only on the *following* action, so the state
recorded on the crashing step still carries the pre-impact speed.

Episodes that never crash contribute `0.0`, so read it alongside `collision_rate`.

In [7]:
for spec in world.get_metric_specs():
    print(f'{spec.name:<24} <- {spec.channel:<20} reduced with {spec.per_episode.value}')

collision_rate           <- crashed              reduced with any
off_road_rate            <- off_road             reduced with any
time_limit_rate          <- time_limit           reduced with any
mean_abs_lane_offset_m   <- abs_lane_offset_m    reduced with mean
mean_speed_mps           <- speed_mps            reduced with mean
collision_speed_mps      <- collision_speed_mps  reduced with max
near_miss_rate           <- near_miss            reduced with any


## Summary

- **State**: identical in both arms — ego pose, speed, Frenet terms, plus fixed relative
  agent slots. Same width in the world and the planner's model.
- **Observation**: the only difference. Absolute kinematics in the baseline; a 12x12
  occupancy grid with no velocities and an 18 m horizon in the POMDP arm.
- **Reward**: lane-centring minus control effort minus collision, normalised to `[0, 1]`
  and zeroed off-road — so off-road is the harsher outcome.

Two caveats to carry into any experiment: the baseline is a near-MDP because driver policy
stays hidden, and the observation window is informative on only ~15% of steps as shipped.